# Model 3 Evaluation: Stable Diffusion Inpainting + LoRA

Complete evaluation workflow for the final epoch-12 checkpoint. This notebook supports paired quantitative evaluation, unpaired holdout evaluation, report generation, gallery export, and Kaggle Dataset publishing.

Run the runtime section once, then choose the paired-test or unpaired-holdout workflow.

## Workflow

```text
Runtime setup and model loading
├── Paired test: cloth_id = person_id → SSIM / PSNR / LPIPS
└── Unpaired holdout: original person-cloth pairs → CLIP similarity / outside-mask MAE
```

Required Kaggle inputs:

- VITON-HD
- DLP cleaned CSVs
- DLP cloth captions
- `/kaggle/input/datasets/khoaanh1234/ckpt-epoch-12-yen`

In [ ]:
from pathlib import Path
from evaluation.model_3_sd_lora.workflow import EVALUATION_ROOT, run_stage

print(f"Model 3 evaluation source: {EVALUATION_ROOT}")

## 1. Dependency Check

Run once at the beginning of a fresh Kaggle session. If packages are installed, restart the session and continue from Section 2.

In [ ]:
run_stage("runtime/00_install_dependencies.py", globals())

## 2. Configure Paths and Validate Checkpoint

Locates all Kaggle inputs, copies the checkpoint to `/kaggle/working/checkpoint_latest`, and verifies that its metadata matches the 17-channel SD + LoRA architecture.

In [ ]:
run_stage("runtime/01_configure_paths.py", globals())

## 3. Load Model Components

Loads the VAE, tokenizer, text encoder, UNet LoRA adapter, expanded `conv_in`, CLIP Vision encoder, Perceiver, Cloth Spatial Projector, and DDIM scheduler.

In [ ]:
run_stage("runtime/02_load_models.py", globals())

## 4. Build Inference Pipeline

Defines `run_inference(person_id, cloth_id, split="test")`. The same function is used for paired and unpaired evaluation.

In [ ]:
run_stage("runtime/03_inference.py", globals())
print("Runtime ready. Choose a workflow below.")

# Workflow A: Paired Test Quantitative Evaluation

Uses every `person_id` from `clean_vto_dataset_test.csv` and forces `cloth_id = person_id`. The generated image is compared with the original person image to calculate SSIM, PSNR, and LPIPS.

The evaluator writes progress after every sample and resumes automatically.

In [ ]:
run_stage("paired_test/01_evaluate_metrics.py", globals())

## Export Paired Three-Column Gallery

Creates `Person Input | Target Cloth | Try-On Result` images from saved predictions without rerunning inference.

In [ ]:
run_stage("paired_test/02_export_three_column_gallery.py", globals())

## Optional: Publish Paired Results

Publishes metrics, summary, chart, and the three-column gallery as a Kaggle Dataset.

In [ ]:
# run_stage("paired_test/03_publish_kaggle_dataset.py", globals())

# Workflow B: Unpaired Holdout Evaluation

Uses the original person-cloth pairs from `holdout_test.csv`. Since these pairs do not have pixel-aligned ground truth, evaluation uses CLIP garment similarity, outside-mask MAE, mask area, inference time, and qualitative comparisons.

In [ ]:
run_stage("unpaired_holdout/01_evaluate_holdout.py", globals())

## Generate Holdout Metric Report and Error Analysis

Ranks samples, summarizes metrics, and exports best/worst qualitative galleries.

In [ ]:
run_stage("unpaired_holdout/02_generate_report.py", globals())

## Optional: Publish Full Holdout Results

In [ ]:
# run_stage("unpaired_holdout/03_publish_kaggle_dataset.py", globals())

# Workflow C: Review Published Holdout Results

These stages are independent of the loaded model. Attach the published unpaired result dataset before running them.

In [ ]:
# run_stage("review_publish/01_review_published_dataset.py", globals())

## Review Manual Shortlist

In [ ]:
# run_stage("review_publish/02_review_manual_shortlist.py", globals())

## Export Report-Ready Shortlist Gallery

In [ ]:
# run_stage("review_publish/03_export_shortlist_gallery.py", globals())